In [1]:
import torch

print("CUDA disponível? ", torch.cuda.is_available())
print("Quantas GPUs?    ", torch.cuda.device_count())
print("Nome da GPU:     ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A")
print("Versão CUDA:     ", torch.version.cuda)
print("Backend cuDNN:   ", torch.backends.cudnn.version())


CUDA disponível?  True
Quantas GPUs?     1
Nome da GPU:      NVIDIA GeForce RTX 3060
Versão CUDA:      12.8
Backend cuDNN:    91002


In [2]:
import numpy as np
import pandas as pd

from pathlib import Path

from ml.embedding import Embedder
from ml.clustering import Clusterer
from papers_lab.io import Storage, PaperTransform
from papers_lab import PaperAnalysis
from eda import Visualizer

/home/kenai/projects/vscode/lisa-llm-reqs/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


UMAP disponível.
HDBSCAN disponível.


In [3]:
root = Path("papers_test")
store = Storage()

researches = []
for sub in ("ieee", "acm"):
    d = root / sub
    if d.exists():
        researches += store.load_dir(d, pattern="*.pkl", recursive=True)

len(researches)
tx = PaperTransform()
df = tx.transform(researches, normalize_keywords=True)

In [4]:
abstracts = df["abstract"].dropna()

emb = Embedder(model_name="sentence-transformers/all-MiniLM-L6-v2")
X = emb.encode(abstracts)
X.shape

(9, 384)

In [5]:
X = np.asarray(X)

clu = Clusterer(method="kmeans", n_clusters=3, reducer="umap")  # PCA fallback automático
labels = clu.fit_predict(X)

print("Resumo:", clu.summary(labels))
print("Silhouette:", clu.silhouette(X, labels))
print("Exemplos por cluster:", clu.exemplars(X, labels, k=3))
#clu.plot_2d(X, labels, title="KMeans nos embeddings")

2025-10-02 21:19:58,328 | INFO | clustering:fit:line68 - Fitting KMeans with n_clusters=3.
2025-10-02 21:19:58,350 | INFO | clustering:fit_predict:line103 - Returning KMeans labels.


Resumo: {0: 1, 1: 3, 2: 5}
Silhouette: 0.026163727045059204
Exemplos por cluster: {0: [3], 1: [5, 8, 7], 2: [1, 2, 4]}


In [6]:
clu_h = Clusterer(method="hdbscan", min_cluster_size=2, reducer="umap")
labels_h = clu_h.fit_predict(X)
print("Resumo HDBSCAN:", clu_h.summary(labels_h))
print("Silhouette HDBSCAN:", clu_h.silhouette(X, labels_h))  # pode retornar None se 1 cluster
clu_h.plot_2d(X, labels_h, title="HDBSCAN nos embeddings")

2025-10-02 21:19:58,373 | INFO | clustering:fit:line82 - Fitting HDBSCAN with min_cluster_size=2.
/home/kenai/projects/vscode/lisa-llm-reqs/.venv/lib/python3.10/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/kenai/projects/vscode/lisa-llm-reqs/.venv/lib/python3.10/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2025-10-02 21:19:58,511 | INFO | clustering:fit_predict:line97 - Returning HDBSCAN labels.
2025-10-02 21:19:58,546 | INFO | clustering:reduce:line52 - Reducing dimensions with UMAP.


Resumo HDBSCAN: {-1: 9}
Silhouette HDBSCAN: None


AttributeError: module 'umap' has no attribute 'UMAP'